In [14]:
import pandas as pd
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load
import datetime

In [ ]:
# df = pd.read_json('tle_data_C06.json')
# selected_data = df[['EPOCH', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2']]

# selected_data.to_csv('sgp4_C06_data.csv', index=False, encoding='utf-8')

In [18]:
df = pd.read_csv('PC38_data.csv')
# df2 = pd.read_csv('sgp4_C06_data.csv')

df['epoch'] = pd.to_datetime(df['epoch'])

df_selected = df[(df['epoch'].dt.hour == 12) & (df['epoch'].dt.minute == 0) & (df['epoch'].dt.second == 0)]
df_selected.to_csv('sp3_C38_2025_onePerDay.csv', index=False, encoding='utf-8')

In [ ]:
# df = pd.read_csv('sgp4_C06_data.csv')

# df['EPOCH'] = pd.to_datetime(df['EPOCH'])
# df_one_per_day = df.groupby(df['EPOCH'].dt.date).first() # DO DOKOŃCZENIA, MA POBIERAĆ JEDEN POMIAR NA DZIEŃ
# df_one_per_day.to_csv('sp4_C06_2025_onePerDay.csv', index=False, encoding='utf-8')

In [ ]:
# # Calculating TLE into TEME

# INPUT_CSV = 'tle_C06_2025_onePerDay.csv'
# OUTPUT_CSV = 'sgp4_TEME_C06_2025_4perRecord.csv'
# L1 = 'TLE_LINE1'
# L2 = 'TLE_LINE2'

# df = pd.read_csv(INPUT_CSV)
# w = []

# day_offset = [0, 1, 3, 5]

# for index, row in df.iterrows():
#     l1 = row[L1]
#     l2 = row[L2]

#     record = row.to_dict()

#     try:
#         # Inicjalizacja modelu SGP4 dla danego TLE
#         sat = Satrec.twoline2rv(l1, l2)
        
#         jd_epoka = sat.jdsatepoch
#         fr_epoka = sat.jdsatepochF
        
#         # Obliczenia dla każdego przesunięcia w dniach
#         for offset in day_offset:
#             jd_cel = jd_epoka + offset
            
#             e, r, v = sat.sgp4(jd_cel, fr_epoka)
            
#             if e == 0:
#                 record[f'X_{offset}d [km]'] = r[0]
#                 record[f'Y_{offset}d [km]'] = r[1]
#                 record[f'Z_{offset}d [km]'] = r[2]
#                 record[f'Vx_{offset}d [km/s]'] = v[0]
#                 record[f'Vy_{offset}d [km/s]'] = v[1]
#                 record[f'Vz_{offset}d [km/s]'] = v[2]
#             else:
#                 # Błąd 
#                 record[f'X_{offset}d [km]'] = None
#                 record[f'Y_{offset}d [km]'] = None
#                 record[f'Z_{offset}d [km]'] = None
#                 record[f'Vx_{offset}d [km/s]'] = None
#                 record[f'Vy_{offset}d [km/s]'] = None
#                 record[f'Vz_{offset}d [km/s]'] = None
                
#     except Exception as ex:
#         print(f"Błąd przetwarzania wiersza {index}: {ex}")
        
#     w.append(record)

# df_wyniki = pd.DataFrame(w)
# df_wyniki.to_csv(OUTPUT_CSV, index=False)

# print(f"Zakończono! Zapisano wyniki do {OUTPUT_CSV}")


Zakończono! Zapisano wyniki do sgp4_TEME_C06_2025_4perRecord.csv


In [17]:
import pandas as pd
import datetime
from skyfield.api import EarthSatellite, load
from skyfield.framelib import itrs

# 1. Konfiguracja
INPUT_CSV = 'tle_C06_2025_onePerDay.csv'
OUTPUT_CSV = 'TLE_to_ECEF_1200_2025.csv'

# PAMIĘTAJ: Zgodnie z naszym wcześniejszym śledztwem, TLE 44204 
# w styczniu 2025 r. odpowiadało satelicie PC38 w plikach SP3!
SAT_ID = 'PC38' 

# Skala czasu dla Skyfield
ts = load.timescale()
df = pd.read_csv(INPUT_CSV) 

w = []

for index, row in df.iterrows():
    l1 = row['TLE_LINE1'] 
    l2 = row['TLE_LINE2'] 
    
    try:
        sat = EarthSatellite(l1, l2, SAT_ID, ts)
        
        # Pobieramy datę wygenerowania TLE jako obiekt datetime (UTC)
        dt_epoka = sat.epoch.utc_datetime()
        
        # WYMUSZAMY godzinę 12:00:00 dla dnia, z którego pochodzi TLE (brak pętli dodającej dni)
        dt_target = dt_epoka.replace(hour=12, minute=0, second=0, microsecond=0)
        
        # Konwersja czasu na format zrozumiały dla Skyfield
        t_cel = ts.from_datetime(dt_target)
        
        # Pobranie pozycji i transformacja do ECEF
        geocentric = sat.at(t_cel)
        pos_ecef = geocentric.frame_xyz(itrs).km
        
        # Formatowanie czasu dla SP3 (zawsze będzie to ... 12:00:00)
        epoch_str = dt_target.strftime('%Y-%m-%d %H:%M:%S')
        
        # Budowanie rekordu
        record = {
            'epoch': epoch_str,
            'sat_id': SAT_ID,
            'x': round(pos_ecef[0], 6),
            'y': round(pos_ecef[1], 6),
            'z': round(pos_ecef[2], 6),
            'clock': 0.0
        }
        
        w.append(record)
            
    except Exception as ex:
        print(f"Błąd dla wiersza {index}: {ex}")

# 3. Zapis do pliku
df_wyniki = pd.DataFrame(w, columns=['epoch', 'sat_id', 'x', 'y', 'z', 'clock'])

# Zabezpieczenie: Jeśli plik CSV miał 2 TLE z tego samego dnia, 
# zostawiamy tylko ostatnie (najświeższe) wyliczenie dla 12:00 tego dnia.
df_wyniki = df_wyniki.drop_duplicates(subset=['epoch'], keep='last')

df_wyniki.to_csv(OUTPUT_CSV, index=False)

print(f"Zakończono! Zapisano zsynchronizowane wyniki do {OUTPUT_CSV}")

Zakończono! Zapisano zsynchronizowane wyniki do TLE_to_ECEF_1200_2025.csv
